In [1]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import sqlite3
from scipy import stats
from data_io import load_from_sqlite

DB_PATH = '../data/processed/wec_bop.db'

clean = load_from_sqlite("SELECT * FROM clean_wec_laps", DB_PATH)

print(f"Loaded: {len(clean):,} rows")
print(f"Columns: {clean.shape[1]}")

Loaded: 334,980 rows
Columns: 43


In [9]:
# For stint analysis — race, target classes, valid_for_stint_analysis
stint_df = clean[
    (clean['valid_for_stint_analysis'] == 1) &
    (clean['class_normalized'].isin(['HYPERCAR', 'GT3']))
].copy()

# For pace analysis — race, target classes, valid_for_pace_analysis
pace_df = clean[
    (clean['valid_for_pace_analysis'] == 1) &
    (clean['class_normalized'].isin(['HYPERCAR', 'GT3']))
].copy()

print(f"Stint df: {len(stint_df):,} rows")
print(f"Pace df:  {len(pace_df):,} rows")
print(f"\nStint df — homologation_group:")
print(stint_df['homologation_group'].value_counts())

Stint df: 111,167 rows
Pace df:  97,941 rows

Stint df — homologation_group:
homologation_group
LMH     44323
GT3     41630
LMDh    25214
Name: count, dtype: int64


In [11]:
def calc_degradation(group: pd.DataFrame) -> float:
    """
    Linear regression lap_time ~ stint_lap.
    Returin slope (sec/lap). Positive = degradation.
    Minimum 4 laps.
    """
    g = group.dropna(subset=['lap_time', 'stint_lap'])
    if len(g) < 4:
        return np.nan
    slope, _, _, _, _ = stats.linregress(g['stint_lap'], g['lap_time'])
    return round(slope, 4)

In [12]:
GROUP_COLS = ['year', 'event', 'session', 'session_id',
              'class_normalized', 'homologation_group',
              'manufacturer', 'car_model_key', 'car',
              'driver_name', 'stint_number']

stint_features = (
    stint_df
    .groupby(GROUP_COLS, as_index=False)
    .agg(
        valid_laps_count  = ('lap_time', 'count'),
        median_lap_time   = ('lap_time', 'median'),
        mean_lap_time     = ('lap_time', 'mean'),
        best_lap_time     = ('lap_time', 'min'),
        std_lap_time      = ('lap_time', 'std'),
        avg_tire_age      = ('est_tire_age', 'mean'),
        wet_stint_flag    = ('is_wet_lap', 'max'),
    )
)

# MAD вручную
mad_vals = (
    stint_df
    .groupby(GROUP_COLS)['lap_time']
    .apply(lambda x: (x - x.median()).abs().median())
    .reset_index(name='mad_lap_time')
)
stint_features = stint_features.merge(mad_vals, on=GROUP_COLS, how='left')

# Деградация
deg_vals = (
    stint_df
    .groupby(GROUP_COLS)
    .apply(calc_degradation, include_groups=False)
    .reset_index(name='stint_degradation')
)
stint_features = stint_features.merge(deg_vals, on=GROUP_COLS, how='left')

# Фильтруем стинты с минимальным числом кругов
from config import MIN_LAPS_PER_STINT
stint_features = stint_features[
    stint_features['valid_laps_count'] >= MIN_LAPS_PER_STINT
].reset_index(drop=True)

print(f"Stint features: {len(stint_features):,} rows")
print(stint_features.head(3))

Stint features: 2,570 rows
   year            event session  session_id class_normalized  \
0  2021  Bahrain 6 Hours    race         698         HYPERCAR   
1  2021  Bahrain 6 Hours    race         698         HYPERCAR   
2  2021  Bahrain 6 Hours    race         698         HYPERCAR   

  homologation_group manufacturer car_model_key  car         driver_name  \
0                LMH       Alpine    Alpine_LMH   36        André NEGRÃO   
1                LMH       Alpine    Alpine_LMH   36  Matthieu Vaxiviere   
2                LMH       Alpine    Alpine_LMH   36  Matthieu Vaxiviere   

   stint_number  valid_laps_count  median_lap_time  mean_lap_time  \
0             3                37          112.917     116.252676   
1             1                25          112.420     112.574720   
2             4                39          113.792     116.647821   

   best_lap_time  std_lap_time  avg_tire_age  wet_stint_flag  mad_lap_time  \
0        111.664     18.753217           0.0        

In [13]:
print("=== stint_features per classes ===")
print(stint_features['class_normalized'].value_counts())

print("\n=== stint_degradation distribution ===")
print(stint_features['stint_degradation'].describe().round(4))

print("\n=== avg_tire_age distribution ===")
print(stint_features['avg_tire_age'].describe().round(1))

=== stint_features per classes ===
class_normalized
HYPERCAR    1450
GT3         1120
Name: count, dtype: int64

=== stint_degradation distribution ===
count    2566.0000
mean        0.3162
std        12.3272
min       -12.9558
25%        -0.0743
50%         0.0168
75%         0.1152
max       620.6080
Name: stint_degradation, dtype: float64

=== avg_tire_age distribution ===
count    2570.0
mean       10.1
std         4.7
min         0.0
25%         5.5
50%        10.7
75%        14.0
max        29.5
Name: avg_tire_age, dtype: float64


In [14]:
def calc_degradation(group: pd.DataFrame) -> float:
    """
    Linear regression lap_time ~ stint_lap.
    Minimum 5 laps.
    """
    g = group.dropna(subset=['lap_time', 'stint_lap'])
    if len(g) < 5:
        return np.nan
    slope, _, _, _, _ = stats.linregress(g['stint_lap'], g['lap_time'])
    return round(slope, 4)

In [15]:
deg = stint_features['stint_degradation'].dropna()
low, high = deg.quantile(0.01), deg.quantile(0.99)
stint_features['stint_degradation'] = stint_features['stint_degradation'].clip(low, high)

print(f"Degradation winsorized: [{low:.4f}, {high:.4f}]")
print(stint_features['stint_degradation'].describe().round(4))

Degradation winsorized: [-2.0582, 3.5359]
count    2566.0000
mean        0.0427
std         0.6027
min        -2.0582
25%        -0.0743
50%         0.0168
75%         0.1152
max         3.5359
Name: stint_degradation, dtype: float64


In [16]:
conn = sqlite3.connect(DB_PATH)
stint_features.to_sql('stint_features', conn, if_exists='replace', index=False)
conn.close()
print(f"Saved {len(stint_features):,} rows → stint_features")

Saved 2,570 rows → stint_features


In [17]:
# Baseline = median all pace-valid laps into
# class_normalized + homologation_group + event + session
baseline = (
    pace_df
    .groupby(['year', 'event', 'session', 'class_normalized', 'homologation_group'])
    ['lap_time']
    .median()
    .reset_index(name='group_baseline')
)

print("=== Baseline sample ===")
print(baseline.head(10).to_string())
print(f"\nTotal baseline rows: {len(baseline):,}")

=== Baseline sample ===
   year            event session class_normalized homologation_group  group_baseline
0  2021  Bahrain 6 Hours    race         HYPERCAR                LMH        113.0140
1  2021  Bahrain 8 Hours    race         HYPERCAR                LMH        111.8500
2  2021            Monza    race         HYPERCAR                LMH         98.7205
3  2021         Portimao    race         HYPERCAR                LMH         92.8010
4  2021              Spa    race         HYPERCAR                LMH        127.2690
5  2022          Bahrain    race         HYPERCAR                LMH        112.5310
6  2022             Fuji    race         HYPERCAR                LMH         92.0750
7  2022          Le Mans    race         HYPERCAR                LMH        212.4075
8  2022            Monza    race         HYPERCAR                LMH         98.9915
9  2022          Sebring    race         HYPERCAR                LMH        111.6995

Total baseline rows: 63


In [18]:
MODEL_GROUP = ['year', 'event', 'session', 'session_id',
               'class_normalized', 'homologation_group',
               'manufacturer', 'car_model_key']

model_pace = (
    pace_df
    .groupby(MODEL_GROUP, as_index=False)
    .agg(
        clean_laps_count = ('lap_time', 'count'),
        median_clean_lap = ('lap_time', 'median'),
        best_clean_lap   = ('lap_time', 'min'),
        std_clean_lap    = ('lap_time', 'std'),
    )
)

print(f"Model pace rows: {len(model_pace):,}")
print(model_pace.head(3))

Model pace rows: 305
   year            event session  session_id class_normalized  \
0  2021  Bahrain 6 Hours    race         698         HYPERCAR   
1  2021  Bahrain 6 Hours    race         698         HYPERCAR   
2  2021  Bahrain 8 Hours    race         705         HYPERCAR   

  homologation_group manufacturer car_model_key  clean_laps_count  \
0                LMH       Alpine    Alpine_LMH               161   
1                LMH       Toyota    Toyota_LMH               315   
2                LMH       Alpine    Alpine_LMH               206   

   median_clean_lap  best_clean_lap  std_clean_lap  
0          113.5830         111.075       0.952732  
1          112.8370         110.860       0.759641  
2          112.8005         110.748       0.877678  


In [19]:
event_model = model_pace.merge(
    baseline,
    on=['year', 'event', 'session', 'class_normalized', 'homologation_group'],
    how='left'
)

# baseline_delta: negative = faster baseline (good for car)
event_model['baseline_delta'] = (
    event_model['median_clean_lap'] - event_model['group_baseline']
)

# consistency_score: normalization CV, higher = more stability
# CV = std/median, consistency = 1/CV normalization 0..1
event_model['cv'] = event_model['std_clean_lap'] / event_model['median_clean_lap']
cv_max = event_model['cv'].quantile(0.95)  # cap for normalization
event_model['consistency_score'] = (
    1 - (event_model['cv'] / cv_max).clip(0, 1)
)

print("=== baseline_delta distribution ===")
print(event_model['baseline_delta'].describe().round(3))

print("\n=== consistency_score distribution ===")
print(event_model['consistency_score'].describe().round(3))

=== baseline_delta distribution ===
count    305.000
mean       0.087
std        0.749
min       -5.672
25%       -0.261
50%        0.008
75%        0.303
max        3.062
Name: baseline_delta, dtype: float64

=== consistency_score distribution ===
count    305.000
mean       0.474
std        0.171
min        0.000
25%        0.408
50%        0.524
75%        0.586
max        0.762
Name: consistency_score, dtype: float64


In [20]:
# Agg stint_features on the level event+model
stint_agg = (
    stint_features
    .groupby(['year', 'event', 'session', 'car_model_key'], as_index=False)
    .agg(
        stints_count   = ('stint_number', 'nunique'),
        long_run_score = ('stint_degradation', 'median'),
        wet_event_flag = ('wet_stint_flag', 'max'),
    )
)

event_model = event_model.merge(
    stint_agg,
    on=['year', 'event', 'session', 'car_model_key'],
    how='left'
)

# wet_vs_dry_delta
wet_dry = (
    pace_df
    .groupby(['year', 'event', 'session', 'car_model_key', 'weather_bucket'])
    ['lap_time']
    .median()
    .unstack('weather_bucket')
    .reset_index()
)
if 'wet' in wet_dry.columns and 'dry' in wet_dry.columns:
    wet_dry['wet_vs_dry_delta'] = wet_dry['wet'] - wet_dry['dry']
else:
    wet_dry['wet_vs_dry_delta'] = np.nan

event_model = event_model.merge(
    wet_dry[['year', 'event', 'session', 'car_model_key', 'wet_vs_dry_delta']],
    on=['year', 'event', 'session', 'car_model_key'],
    how='left'
)

print(f"Event model features: {len(event_model):,} rows")
print(f"Columns: {event_model.columns.tolist()}")

Event model features: 305 rows
Columns: ['year', 'event', 'session', 'session_id', 'class_normalized', 'homologation_group', 'manufacturer', 'car_model_key', 'clean_laps_count', 'median_clean_lap', 'best_clean_lap', 'std_clean_lap', 'group_baseline', 'baseline_delta', 'cv', 'consistency_score', 'stints_count', 'long_run_score', 'wet_event_flag', 'wet_vs_dry_delta']


In [21]:
# std baseline_delta per track for every model+season
# Low std = stability model for every tracks
track_stability = (
    event_model
    .groupby(['year', 'class_normalized', 'homologation_group', 'car_model_key'])
    ['baseline_delta']
    .std()
    .reset_index(name='track_balance_score')
)

event_model = event_model.merge(
    track_stability,
    on=['year', 'class_normalized', 'homologation_group', 'car_model_key'],
    how='left'
)

print("=== track_balance_score distribution ===")
print(event_model['track_balance_score'].describe().round(3))

=== track_balance_score distribution ===
count    305.000
mean       0.458
std        0.345
min        0.096
25%        0.259
50%        0.346
75%        0.596
max        2.247
Name: track_balance_score, dtype: float64


In [22]:
conn = sqlite3.connect(DB_PATH)
event_model.to_sql('event_model_features', conn, if_exists='replace', index=False)
conn.close()
print(f"Saved {len(event_model):,} rows → event_model_features")

Saved 305 rows → event_model_features


In [24]:
print("=== TOP 5 models per avg baseline_delta (all years) ===")
print("(negative = faster baseline = good)")

summary = (
    event_model
    .groupby(['class_normalized', 'homologation_group', 'car_model_key'])
    .agg(
        events        = ('event', 'nunique'),
        avg_delta     = ('baseline_delta', 'mean'),
        avg_consist   = ('consistency_score', 'mean'),
        avg_long_run  = ('long_run_score', 'median'),
    )
    .round(3)
    .sort_values('avg_delta')
    .reset_index()
)

for cls in ['HYPERCAR', 'GT3']:
    print(f"\n--- {cls} ---")
    subset = summary[summary['class_normalized'] == cls]
    print(subset[['homologation_group', 'car_model_key',
                  'events', 'avg_delta',
                  'avg_consist', 'avg_long_run']].to_string(index=False))

=== TOP 5 models per avg baseline_delta (all years) ===
(negative = faster baseline = good)

--- HYPERCAR ---
homologation_group        car_model_key  events  avg_delta  avg_consist  avg_long_run
               LMH           Toyota_LMH      13     -0.500        0.579         0.020
               LMH          Ferrari_LMH      10     -0.449        0.553         0.010
              LMDh        Cadillac_LMDh      10     -0.079        0.508        -0.008
              LMDh         Porsche_LMDh      10     -0.007        0.517         0.008
              LMDh             BMW_LMDh       8      0.144        0.532         0.012
               LMH           Alpine_LMH      13      0.208        0.507         0.008
               LMH          Peugeot_LMH      11      0.232        0.562         0.011
              LMDh     Lamborghini_LMDh       6      0.289        0.573         0.026
               LMH     Aston Martin_LMH       8      0.698        0.465        -0.008
               LMH      Glicke

## 📊 Выводы по Feature Engineering

### HYPERCAR

**LMH vs LMDh баланс:**
Toyota_LMH и Ferrari_LMH системно быстрее своего baseline на **-0.50** и **-0.45 сек** соответственно
на протяжении 10–13 событий — это устойчивое преимущество, а не случайный результат.
Porsche_LMDh и Cadillac_LMDh держатся около нуля (-0.007 и -0.079), что говорит о хорошем
балансе внутри LMDh группы.

**Аутсайдеры:**
Vanwall (+2.42 сек) и Isotta Fraschini (+2.14 сек) значительно медленнее baseline,
однако оба производителя имеют малое число событий (5 и 3), поэтому их результаты
следует интерпретировать с низким confidence.

**Деградация:**
Большинство моделей показывают деградацию близкую к нулю (0.008–0.020 сек/круг),
что типично для современных GT и Hypercar шин на длинных стинтах WEC.
Isotta Fraschini выбивается с 0.060 сек/круг — возможно связано с малой выборкой
или реальными проблемами с управлением шинами.

---

### GT3

**Плотность поля:**
Разброс `avg_baseline_delta` составляет всего **~0.6 секунды** от лучшего (Lamborghini -0.31)
до худшего (Ford +0.29) — это хороший индикатор работающего BoP в классе LMGT3.

**Лидеры:**
Lamborghini, Aston Martin и Ferrari показывают небольшое преимущество над baseline.
При этом у Lamborghini только 6 событий (класс появился в 2024), что снижает
надёжность оценки.

**Деградация в GT3 выше:**
Средняя деградация в GT3 (0.014–0.062 сек/круг) заметно выше чем в HYPERCAR,
что типично для GT3 шин на более длинных стинтах.


---

### Ограничения текущего этапа

- `wet_vs_dry_delta` не посчитан для большинства моделей из-за малого числа wet-событий
- `track_balance_score` требует минимум 3–4 событий для надёжной оценки
- GT3 данные только за 2 сезона (2024–2025) — ограничивает cross-season анализ
- Производители с менее чем 5 событиями (Isotta Fraschini, Vanwall, Lamborghini LMDh)
  получат сниженный `confidence_score` в Recommender

---


ENGLISH:

## 📊 Conclusions on Feature Engineering

### HYPERCAR

**LMH vs LMDh balance:**
Toyota_LMH and Ferrari_LMH are consistently faster than their baseline by -0.50 and -0.45 sec respectively
across 10–13 events — this represents a stable advantage rather than random variation.
Porsche_LMDh and Cadillac_LMDh remain close to zero (-0.007 and -0.079), indicating good
balance within the LMDh group.

**Outliers:**
Vanwall (+2.42 sec) and Isotta Fraschini (+2.14 sec) are significantly slower than the baseline,
however both manufacturers have a small number of events (5 and 3), so their results
should be interpreted with low confidence.

**Degradation:**
Most models show near-zero degradation (0.008–0.020 sec/lap),
which is typical for modern GT and Hypercar tires over long WEC stints.
Isotta Fraschini stands out with 0.060 sec/lap — possibly due to small sample size
or real tire management issues.

### GT3

**Field density:**
The spread of `avg_baseline_delta` is only ~0.6 seconds from best (Lamborghini -0.31)
to worst (Ford +0.29) — a strong indicator of effective BoP in the LMGT3 class.

**Leaders:**
Lamborghini, Aston Martin, and Ferrari show a slight advantage over the baseline.
However, Lamborghini only has 6 events (the class was introduced in 2024),
which reduces the reliability of the estimate.

**Higher degradation in GT3:**
Average degradation in GT3 (0.014–0.062 sec/lap) is noticeably higher than in HYPERCAR,
which is typical for GT3 tires over longer stints.

---

### Limitations of the current stage


- `wet_vs_dry_delta` is not calculated for most models due to the low number of wet events
- `track_balance_score` requires at least 3–4 events for a reliable estimate
- GT3 data covers only 2 seasons (2024–2025), which limits cross-season analysis
- Manufacturers with fewer than 5 events (Isotta Fraschini, Vanwall, Lamborghini LMDh)
  will receive a reduced `confidence_score` in the Recommender


---